# 04 - Qwen2.5-VL-3B on POPE (3,000) and AMBER (3,000)  (Google Colab, GPU)

**GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY.** Do not run this on a laptop. It was not run on a GPU by whoever wrote it; the CPU-side logic it calls (`src/extract_features_multi.py`) is unit-tested, the model loading is not.

It runs the *same frozen protocol* as notebooks 01 and 03 (prompt `"{question} Answer yes or no."`, last prompt token, every layer, forced yes/no scoring, image-level 70/15/15 split) for `qwen25vl-3b`, first on POPE (the same 3,000-question seeded subset every other model uses) and then on AMBER (1,000 queries per subset). Each dataset goes through a **smoke test with an automatic gate**: the full run only starts if `answer_mass` (P(yes)+P(no)) is close to 1.

## Colab guide
1. **Runtime:** *Runtime > Change runtime type > GPU* (T4 or better; an L4/A100 is much faster). Keep **bfloat16** (the registry notes that fp16 can overflow for this model); a T4 emulates bf16 and is slow.
2. **Get the code:** the setup cell clones the repo (branch in the config). If the clone fails (private repo / branch not pushed), upload the repo folder to `/content/SC4001Assignment_2026`.
3. **Run the cells in order.** Set `USE_DRIVE = True` so outputs survive disconnects. Time: POPE and AMBER are 3,000 questions each. The Qwen2-VL-2B run took about 1.9 s/question on a T4; this 3B model will be slower, so budget **several hours per dataset on a free T4**. The smoke test prints a time estimate; if it does not fit your session, run one dataset per session (set `DATASETS = ["pope"]`, then `["amber"]`).
4. **After a disconnect:** re-run all cells from the top with the same config; finished 100-question shards are skipped.
5. **Outputs** (in `OUT_DIR`): `pope_qwen25vl-3b_{features.pt, metadata.parquet, run_config.json}` and `amber_qwen25vl-3b_...`. The `.pt` files are about `3000 x (layers+1) x hidden x 2 bytes` each (roughly 0.4-0.5 GB; the printed shape tells you the real numbers). **Never commit the `.pt` files.**
6. **Bring them back:** copy the six files from Drive into your local repo's `results/cached_features/`, then run the CPU evaluation (last cell prints the command).
7. **Verify:** each full run ends with a verification printout; it must show 0 missing rows, 0 duplicates, 0 non-finite values, `answer_mass` mean close to 1 and **no `!!! FLAG` line**.
8. **OOM:** *Runtime > Restart session*, re-run from the top (it resumes).

## Config

In [ ]:
MODEL_KEY = "qwen25vl-3b"       # fixed for this notebook (see src/vlm_registry.py)
DATASETS = ["pope", "amber"]    # run order; use one entry per session if time is short
POPE_SUBSET_SIZE = 3000         # same seeded first-3000 questions as the other models (needed for the paired cross-model comparison)
AMBER_PER_SUBSET = 1000         # random queries per AMBER subset (existence / attribute / relation)
SMOKE_N = 20                    # questions in the smoke test
MIN_ANSWER_MASS = 0.9           # the full run is only started if the smoke test's mean answer_mass is at least this
LAYERS = "all"                  # keep "all": the other models' caches use every layer
REPO_URL = "https://github.com/RUANLASS/SC4001Assignment_2026.git"
BRANCH = "main"                 # use "extra-models-amber" if that branch has not been merged on GitHub yet
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/sc4001_extra"
AMBER_HF_IMAGES = None          # fallback if the Google-Drive image download fails: "visual-preference/AMBER"

## a. Setup  (GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY)

In [ ]:
import os, re, sys, subprocess
from pathlib import Path
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
if not Path("src/extract_features_multi.py").exists():
    target = Path("/content/SC4001Assignment_2026")
    if not target.exists():
        r = subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(target)], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert (target / "src" / "extract_features_multi.py").exists(), "clone failed (private repo? wrong BRANCH?): upload the repo folder instead"
    os.chdir(target)
print("working dir:", os.getcwd())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-extra.txt"])
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU, then re-run"
OUT_DIR = "results/cached_features"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); OUT_DIR = DRIVE_DIR; os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", OUT_DIR)
print(subprocess.run([sys.executable, "src/extract_features_multi.py", "--list-models"], capture_output=True, text=True).stdout)

## b. AMBER data (only if "amber" is in DATASETS)
Metadata and ground truth come from the AMBER GitHub repo, pixels from the Google-Drive zip linked in its README (or from the Hugging Face mirror if `AMBER_HF_IMAGES` is set).

In [ ]:
if "amber" in DATASETS:
    if not Path("data/amber_repo").exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/junyangwang0410/AMBER", "data/amber_repo"])
    have_images = Path("data/amber_images").exists() and any(Path("data/amber_images").rglob("AMBER_*.jpg"))
    if AMBER_HF_IMAGES is None and not have_images:
        import gdown
        gdown.download(id="1MaCHgtupcZUjf007anNl4_MV0o4DjXvl", output="data/amber_images.zip", quiet=False)
        subprocess.check_call(["unzip", "-q", "-o", "data/amber_images.zip", "-d", "data/amber_images"])
AMBER_ARGS = ["--amber-hf-images", AMBER_HF_IMAGES] if AMBER_HF_IMAGES else []

## c. Smoke test with automatic gate, then the full run  (GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY)
For each dataset: (1) smoke test on `SMOKE_N` questions, nothing is written; (2) if the mean `answer_mass` is below `MIN_ANSWER_MASS` the notebook **stops** (the model is not answering yes/no with the frozen prompt: check the printed token ids and prompt); (3) otherwise the resumable full run starts (checkpoint every 100 questions).

In [ ]:
def run_cmd(args):
    """Run a command, echo its output live, return (exit code, full output)."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line)
    p.wait()
    return p.returncode, "".join(lines)

def common_args(ds):
    a = ["--model", MODEL_KEY, "--dataset", ds, "--layers", LAYERS, "--out-dir", OUT_DIR, "--per-subset", str(AMBER_PER_SUBSET)]
    if ds == "pope":
        a += ["--subset-size", str(POPE_SUBSET_SIZE)]
    return a + (AMBER_ARGS if ds == "amber" else [])

for ds in DATASETS:
    print(f"\n{'=' * 30} {MODEL_KEY} on {ds}: smoke test {'=' * 30}")
    code, out = run_cmd([sys.executable, "src/extract_features_multi.py", *common_args(ds), "--smoke", str(SMOKE_N)])
    m = re.search(r"mean answer_mass ([0-9.]+)", out)
    assert code == 0 and m, "smoke test failed (see output above); do not start the full run"
    mass = float(m.group(1))
    assert mass >= MIN_ANSWER_MASS, f"answer_mass {mass:.3f} < {MIN_ANSWER_MASS}: STOP. The model is not answering yes/no; check the yes/no token ids and prompt above."
    print(f"\nanswer_mass {mass:.3f} OK -> starting the full {ds} run")
    code, out = run_cmd([sys.executable, "src/extract_features_multi.py", *common_args(ds)])
    assert code == 0, "full run failed (see output above); re-run this cell to resume"
    assert "!!! FLAG" not in out, "verification raised a FLAG (see output above); do not use these files until it is understood"

## d. Files to bring back

In [ ]:
print(subprocess.run(["ls", "-la", OUT_DIR], capture_output=True, text=True).stdout)
print("Copy the *_qwen25vl-3b_* files into your local results/cached_features/ (never commit the .pt files), then on your laptop run e.g.:")
print("  python src/probe_transfer_eval.py --models qwen2vl-2b smolvlm-500m qwen25vl-3b --probe logreg --out-dir results/final/cross_model_logreg")